# Getting Started with Coppuccino

This notebook demonstrates the core workflow:
1. Generate synthetic correlated data
2. Fit a copula normalizing flow
3. Sample from the fitted flow and compare
4. Evaluate log probabilities
5. Save and load the model

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from coppuccino import normalizing_flows_fit, sample, log_prob, save_flow, load_flow

## 1. Generate synthetic data

We create a 4D correlated dataset that mimics the kind of structure
you'd see in MCMC posterior samples.

In [ ]:
np.random.seed(42)

n_samples = 5000
params = ["x1", "x2", "x3", "x4"]

# Correlated Gaussian
mean = [0, 1, -0.5, 2]
cov = [
    [1.0, 0.7, -0.3, 0.1],
    [0.7, 1.5,  0.2, 0.4],
    [-0.3, 0.2, 0.8, -0.5],
    [0.1, 0.4, -0.5, 1.2],
]
data = np.random.multivariate_normal(mean, cov, n_samples)

# Add some non-Gaussianity: skew x3
data[:, 2] = np.exp(data[:, 2])

print(f"Data shape: {data.shape}")

## 2. Fit a copula normalizing flow

In [ ]:
flow = normalizing_flows_fit(data, max_epochs=200, rng_seed=42)

## 3. Sample from the flow and compare marginals

In [ ]:
flow_samples = sample(flow, n_samples=n_samples, rng_seed=123)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for i, ax in enumerate(axes):
    ax.hist(data[:, i], bins=40, density=True, alpha=0.5, label="Data")
    ax.hist(flow_samples[:, i], bins=40, density=True, alpha=0.5, label="Flow")
    ax.set_title(params[i])
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Compare pairwise correlations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [(0, 1), (1, 2), (2, 3)]
for ax, (i, j) in zip(axes, pairs):
    ax.scatter(data[:, i], data[:, j], alpha=0.1, s=3, label="Data")
    ax.scatter(flow_samples[:, i], flow_samples[:, j], alpha=0.1, s=3, label="Flow")
    ax.set_xlabel(params[i])
    ax.set_ylabel(params[j])
    ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## 5. Evaluate log probabilities

In [ ]:
log_probs = log_prob(flow, data[:100])
print(f"Mean log prob on training data: {np.mean(log_probs):.3f}")
print(f"Std log prob: {np.std(log_probs):.3f}")

## 6. Save and load the model

In [ ]:
save_flow(flow, "example_flow.pkl")
loaded_flow = load_flow("example_flow.pkl")

# Verify loaded flow produces the same samples
loaded_samples = sample(loaded_flow, n_samples=1000, rng_seed=99)
original_samples = sample(flow, n_samples=1000, rng_seed=99)
print(f"Max difference between original and loaded samples: {np.max(np.abs(loaded_samples - original_samples)):.2e}")